![iceberg-logo](https://www.apache.org/logos/res/iceberg/iceberg.png)

In [ ]:
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions.*

val spark = SparkSession.builder().appName("Jupyter").getOrCreate()

spark.version()

## Load NYC Taxi/Limousine Trip Data

For this notebook, we will use the New York City Taxi and Limousine Commision Trip Record Data that's available on the AWS Open Data Registry. This contains data of trips taken by taxis and for-hire vehicles in New York City. We'll save this into an iceberg table called `taxis`.

First, load the Parquet file using Spark DataFrames:

In [ ]:
val tbl_taxis = spark.read().parquet("/home/iceberg/data/yellow_tripdata_2021-04.parquet")
tbl_taxis

## Creating the table

Next, create the namespace, and the `taxis` table from the schema that's derived from the Arrow schema:

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS default")

In [ ]:
spark.sql("DROP TABLE IF EXISTS default.taxis")
tbl_taxis.limit(0).write().format("iceberg").mode("overwrite").saveAsTable("default.taxis")

spark.table("default.taxis")

## Write the actual data into the table

This will create a new snapshot on the table:

In [ ]:
spark.sql("DELETE FROM default.taxis WHERE true")
tbl_taxis.writeTo("default.taxis").append()

spark.table("default.taxis")

## Append more data

Let's append another month of data to the table:

In [ ]:
val may_taxis = spark.read().parquet("/home/iceberg/data/yellow_tripdata_2021-05.parquet")
may_taxis.writeTo("default.taxis").append()

spark.table("default.taxis")

## Load data into a Spark DataFrame

We'll fetch the table using the REST catalog that comes with the setup.

In [ ]:
val df = spark.sql("""
SELECT *
FROM default.taxis
WHERE tpep_pickup_datetime >= TIMESTAMP '2021-05-01 00:00:00'
""".trimIndent())

In [ ]:
df.cache()
df

In [ ]:
df.count()

In [ ]:
df.printSchema()
df.describe("fare_amount").show()

In [ ]:
df.show(20, false)

In [ ]:
df.select("fare_amount").summary("count", "min", "25%", "50%", "75%", "max").show()

In [ ]:
val fare_stats = df.agg(
    avg("fare_amount").alias("fare_mean"),
    stddev_pop("fare_amount").alias("fare_stddev")
).first()

val fare_mean = fare_stats.getAs<Double>("fare_mean")
val fare_stddev = fare_stats.getAs<Double>("fare_stddev")

val cleaned_df = df
    .filter(col("fare_amount") > lit(0.0))
    .filter(abs(col("fare_amount") - lit(fare_mean)) / lit(fare_stddev) < lit(3.0))

In [ ]:
cleaned_df.select("fare_amount").summary("count", "min", "25%", "50%", "75%", "max").show()

# Spark SQL

Use Spark SQL to query the DataFrame directly.

In [ ]:
df.createOrReplaceTempView("df")
cleaned_df.createOrReplaceTempView("cleaned_df")

In [ ]:
spark.sql("SELECT * FROM df LIMIT 20").show()

In [ ]:
val tip_amount = spark.sql("""
SELECT tip_amount
FROM df
""".trimIndent())

tip_amount.show()

In [ ]:
tip_amount.summary("count", "min", "25%", "50%", "75%", "max").show()

In [ ]:
val tip_amount_filtered = spark.sql("""
WITH tip_amount_stddev AS (
    SELECT STDDEV_POP(tip_amount) AS tip_amount_stddev
    FROM df
)
SELECT tip_amount
FROM df, tip_amount_stddev
WHERE tip_amount > 0
  AND tip_amount < tip_amount_stddev * 3
""".trimIndent())

tip_amount_filtered.show()

In [ ]:
tip_amount_filtered.summary("count", "min", "25%", "50%", "75%", "max").show()

# Iceberg with Spark SQL

This notebook shows how you can load data into Spark DataFrames and query it directly with Spark SQL. Iceberg allows you to take a slice out of the data that you need for your analysis while reducing scan time and memory use.

In [ ]:
spark.table("default.taxis").count()